In [1]:
import torch

torch.manual_seed(42)

batch_size = 4
seq_len = 6
input_size = 3
hidden_size = 5

X = torch.randn(batch_size, seq_len, input_size)

print("Input shape:", X.shape)

Input shape: torch.Size([4, 6, 3])


In [2]:
def lstm_cell(x, h, c, W, b):

    gates = torch.cat([x, h], dim=1) @ W + b

    i, f, o, g = torch.chunk(gates, 4, dim=1)

    i = torch.sigmoid(i)
    f = torch.sigmoid(f)
    o = torch.sigmoid(o)
    g = torch.tanh(g)

    c = f * c + i * g
    h = o * torch.tanh(c)

    return h, c

In [3]:
def init_params():

    W = torch.randn(
        input_size + hidden_size,
        4 * hidden_size
    ) * 0.1

    b = torch.zeros(4 * hidden_size)

    return W, b


W_f, b_f = init_params()
W_b, b_b = init_params()

In [4]:
def forward_lstm(X, W, b):

    h = torch.zeros(X.size(0), hidden_size)
    c = torch.zeros_like(h)

    outputs = []

    for t in range(seq_len):

        h, c = lstm_cell(
            X[:, t, :], h, c, W, b
        )

        outputs.append(h)

    return torch.stack(outputs, dim=1)

In [5]:
def backward_lstm(X, W, b):

    h = torch.zeros(X.size(0), hidden_size)
    c = torch.zeros_like(h)

    outputs = []

    for t in reversed(range(seq_len)):

        h, c = lstm_cell(
            X[:, t, :], h, c, W, b
        )

        outputs.append(h)

    outputs.reverse()

    return torch.stack(outputs, dim=1)

In [6]:
forward_out = forward_lstm(X, W_f, b_f)
backward_out = backward_lstm(X, W_b, b_b)

output = torch.cat(
    [forward_out, backward_out],
    dim=2
)

print("Forward output:", forward_out.shape)
print("Backward output:", backward_out.shape)
print("BiLSTM output:", output.shape)

Forward output: torch.Size([4, 6, 5])
Backward output: torch.Size([4, 6, 5])
BiLSTM output: torch.Size([4, 6, 10])


In [7]:
print("BiLSTM implemented successfully from scratch.")
print("No nn.RNN or nn.LSTM was used.")

BiLSTM implemented successfully from scratch.
No nn.RNN or nn.LSTM was used.
